# 어태치먼트 시스템 데모

Claude Code 의 30+ 종 자동 첨부 메커니즘을 베이스 골격으로 포팅 — Protocol + 3그룹 분류 + `<system-reminder>` 자동 wrap + assistant turn 카운터 + smoosh + 도구 풀 게이트 슬롯 + 베이스 디폴트 어태치먼트 2종 (`date_change`, `todo_reminder`) + LLM 호출 helper β.

## 작업순서
1. **Setup** — sys.path patch + import + `attachment_registry` snapshot/restore (대화형 격리)
2. **1부: 베이스 사용 흐름** — 빈 ctx 호출 + `date_change`/`todo_reminder` 즉시 시연
3. **2부: register/override** — 도메인 Attachment 만들기 + register + 베이스 디폴트 override
4. **3부: escape hatch + 위험 가드** — 도구 풀 게이트 / `is_subagent` MAIN_THREAD 격리 / 1초 timeout
5. **Cleanup** — registry snapshot 복원
6. **다른 모듈 연계** — `call_with_attachments` + LLM 클라이언트 통합 mock + NFR-2 정적 해시 invariant 시연
7. **실습 4개**

## Setup

`uv` 가 정식 ipykernel 등록 전까지 `sys.path` patch 로 임시. 정식 등록 권장: `uv add --dev jupyter ipykernel && uv run python -m ipykernel install --user --name best-agent-base` (사용자 승인 필요).

**환경 변수 불필요** — 본 노트북은 mock LLMClient 사용. 실제 호출은 `main.py` 참조.

In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from best_agent_base.attachments.builtins import date_change as _dc  # auto-register
from best_agent_base.attachments.builtins import todo_reminder as _tr  # auto-register
from best_agent_base.attachments.collect import collect_attachments
from best_agent_base.attachments.counter import count_turns_since
from best_agent_base.attachments.gates import register_tool_pool_gate, _gates
from best_agent_base.attachments.integrate import call_with_attachments
from best_agent_base.attachments.protocol import Attachment, AttachmentGroup
from best_agent_base.attachments.registry import attachment_registry
from best_agent_base.attachments.smoosh import smoosh_into_last_tool_result
from best_agent_base.llm.client import LLMResponse, TokenUsage
from best_agent_base.messages import Message, TextBlock, ThinkingBlock, ToolResultBlock, ToolUseBlock
from best_agent_base.prompts.render import RenderContext, get_static_hash

_ = (_dc, _tr)  # auto-register 의도, lint silence

# 노트북 격리 — 작업 시작 전 attachment_registry / gates 스냅샷
_REGISTRY_SNAPSHOT = dict(attachment_registry._attachments)
_GATES_SNAPSHOT = {k: list(v) for k, v in _gates.items()}

def restore_state():
    """노트북 cleanup — 베이스 디폴트만 남김."""
    attachment_registry._attachments.clear()
    attachment_registry._attachments.update(_REGISTRY_SNAPSHOT)
    _gates.clear()
    _gates.update(_GATES_SNAPSHOT)
    print(f'restored — registry size={len(attachment_registry._attachments)}, gates={len(_gates)}')

print(f'베이스 디폴트 어태치먼트: {sorted(attachment_registry._attachments.keys())}')

## 1부: 베이스 사용 흐름

`collect_attachments(ctx, *, user_input)` 한 줄로 3그룹 어태치먼트 수집 + null 필터 + `<system-reminder>` 자동 wrap.

베이스에는 디폴트 2종 (`date_change`, `todo_reminder`) 이 자동 등록되어 있어 별도 구현 없이 즉시 시연 가능.

In [ ]:
import asyncio

# 1.1 — 빈 ctx (last_emit_date None, messages 빈 튜플) → 어태치먼트 0개
msgs = await collect_attachments(RenderContext(), user_input='hello')
print(f'1.1 빈 ctx — collected {len(msgs)}건')

In [ ]:
# 1.2 — date_change 트리거: yesterday 박아서 자정 감지
yesterday = date.today() - timedelta(days=1)
ctx_date = RenderContext(last_emit_date=yesterday)
msgs = await collect_attachments(ctx_date, user_input='안녕')
print(f'1.2 date_change — collected {len(msgs)}건')
for m in msgs:
    print(m.content[0].text)

In [ ]:
# 1.3 — todo_reminder 트리거: 10 assistant 메시지 동안 TodoWrite 호출 없음
fake_msgs = tuple(
    Message(role='assistant', content=(TextBlock(text=f'r{i}'),)) for i in range(10)
)
ctx_todo = RenderContext(messages=fake_msgs, todos=('a', 'b'))
msgs = await collect_attachments(ctx_todo, user_input='진행해')
print(f'1.3 todo_reminder — collected {len(msgs)}건')
for m in msgs:
    print(m.content[0].text[:200], '...')

In [ ]:
# 1.4 — 이중 호출 시그니처: user_input=None → USER_INPUT 그룹 자동 스킵 (in-loop ReAct 후)
msgs_user_turn = await collect_attachments(ctx_date, user_input='hi')
msgs_in_loop   = await collect_attachments(ctx_date, user_input=None)
print(f'1.4 user-turn entry: {len(msgs_user_turn)}건 / in-loop: {len(msgs_in_loop)}건')
# 베이스 디폴트는 ALL_THREAD 라 두 케이스 동일 — 도메인 USER_INPUT 어태치먼트 추가 시 차이 발생

## 2부: register/override — 도메인 어태치먼트 끼우기

`Attachment(Protocol)` 만족하는 클래스 만들고 `attachment_registry.register(name, instance)` 한 줄로 끝.

베이스 0줄 수정 (Open/Closed).

In [ ]:
# 2.1 — 도메인 USER_INPUT 어태치먼트 (사용자 텍스트 의존)
class _MentionExpander:
    """`@foo` 멘션 보이면 placeholder 컨텍스트 반환."""
    name = 'mention_expander'
    group = AttachmentGroup.USER_INPUT

    async def build(self, ctx: RenderContext) -> str | None:
        # 본 데모는 직접 user_input 못 받음 — 도메인 ctx 슬롯에 맡기는 게 더 일반적이지만
        # 여기서는 ctx.tool_pool 의 첫 항목을 placeholder 로
        if not ctx.tool_pool:
            return None
        return f'mention placeholder — pool 첫 도구: {sorted(ctx.tool_pool)[0]}'

attachment_registry.register('mention_expander', _MentionExpander())
ctx_with_pool = RenderContext(tool_pool=frozenset({'Read', 'Edit'}))
msgs = await collect_attachments(ctx_with_pool, user_input='@foo')
print(f'2.1 도메인 register — collected {len(msgs)}건')
for m in msgs:
    print(m.content[0].text)

In [ ]:
# 2.2 — 베이스 디폴트 override: date_change 를 도메인 버전으로 교체
class _MyDateChange:
    name = 'date_change'
    group = AttachmentGroup.ALL_THREAD

    async def build(self, ctx: RenderContext) -> str | None:
        if ctx.last_emit_date is None or ctx.last_emit_date == date.today():
            return None
        return '🌞 새로운 하루 시작! (도메인 커스텀 메시지)'

attachment_registry.override('date_change', _MyDateChange())
msgs = await collect_attachments(ctx_date, user_input='hi')
print(f'2.2 override — collected {len(msgs)}건')
for m in msgs:
    print(m.content[0].text)

## 3부: escape hatch + 위험 가드

도메인이 베이스 동작을 "안 작동시킬" 수 있는 다양한 슬롯 + 1초 timeout 안전망.

In [ ]:
# 3.1 — 도구 풀 게이트로 todo_reminder 비활성
register_tool_pool_gate('todo_reminder', lambda tools: 'EmergencyAlert' in tools)

ctx_with_emergency = RenderContext(
    messages=fake_msgs,
    todos=('a',),
    tool_pool=frozenset({'EmergencyAlert', 'Read'}),
)
msgs_gated = await collect_attachments(ctx_with_emergency, user_input='hi')
print(f'3.1 게이트 활성 — collected {len(msgs_gated)}건 (todo_reminder 스킵 기대)')

In [ ]:
# 3.2 — is_subagent: MAIN_THREAD 그룹 자동 제외 (서브에이전트 격리, 원칙 #7)
class _MainThreadOnly:
    name = 'ide_selection_demo'
    group = AttachmentGroup.MAIN_THREAD

    async def build(self, ctx):
        return '메인 전용 — IDE 선택 영역 placeholder'

attachment_registry.register('ide_selection_demo', _MainThreadOnly())

main_msgs = await collect_attachments(RenderContext(), user_input='x')
sub_msgs  = await collect_attachments(RenderContext(is_subagent=True), user_input='x')
print(f'3.2 main: {len(main_msgs)}건 / subagent: {len(sub_msgs)}건 (MAIN_THREAD 자동 제외)')

In [ ]:
# 3.3 — 1초 timeout: 느린 어태치먼트 잘림, 정상은 보존
class _Slow:
    name = 'slow_demo'
    group = AttachmentGroup.ALL_THREAD

    async def build(self, ctx):
        await asyncio.sleep(2.0)
        return 'should never appear'

attachment_registry.register('slow_demo', _Slow())

import time
started = time.monotonic()
msgs = await collect_attachments(RenderContext(), user_input='hi')
elapsed = time.monotonic() - started
print(f'3.3 timeout — collected {len(msgs)}건, elapsed={elapsed:.2f}s (1초대 + 정상만)')

In [ ]:
# 3.4 — counter / smoosh 헬퍼 단독 사용
msgs_history = (
    Message(role='assistant', content=(TextBlock(text='a'),)),
    Message(role='assistant', content=(ThinkingBlock(text='reason'),)),  # thinking-only 제외
    Message(role='assistant', content=(TextBlock(text='b'),)),
)
n = count_turns_since(msgs_history, lambda m: False)  # 매치 없음 → 전체 카운트
print(f'3.4 count_turns_since (thinking 제외): {n} (기대 2)')

msgs_with_tool = (
    Message(role='user', content=(ToolResultBlock(tool_use_id='toolu_1', content='ok'),)),
)
smooshed = smoosh_into_last_tool_result(msgs_with_tool, '<system-reminder>x</system-reminder>')
print(f'smoosh — last user content blocks: {len(smooshed[-1].content)} (기대 2 — tool_result + reminder)')

## Cleanup

노트북 작업 중 등록한 어태치먼트 + 게이트 모두 복원 — 베이스 디폴트만 남김.

In [ ]:
restore_state()
print(f'cleanup 완료 — registry: {sorted(attachment_registry._attachments.keys())}')

## 다른 모듈 연계

본 모듈은 두 번째로 큰 invariant 보장: **시스템 프롬프트 정적/동적 분리의 BOUNDARY 위 정적 7섹션 해시는 어태치먼트 등록·발화에 무영향** → LLM 클라이언트 통합의 KV 캐시 자동 유지.

또한 **LLM 클라이언트 통합** 의 `LLMClient` Protocol 시그니처 무변경 (`generate(ctx, *, cache_policy)`) — `call_with_attachments` helper 가 새 ctx 만들어 호출.

In [ ]:
# NFR-2 invariant 시연 — get_static_hash 동일 (어태치먼트 등록·발화 무영향)
h_before = get_static_hash(RenderContext())

class _Loud:
    name = 'loud_test'
    group = AttachmentGroup.ALL_THREAD
    async def build(self, ctx):
        return 'noisy ' * 100

attachment_registry.register('loud_test', _Loud())
_ = await collect_attachments(RenderContext(), user_input='x')
h_after = get_static_hash(RenderContext())

print(f'before: {h_before}')
print(f'after : {h_after}')
print(f'동일? {h_before == h_after}  ← LLM 클라이언트 통합의 KV 캐시 invariant 자동 보존')
restore_state()

In [ ]:
# call_with_attachments helper β — Phase 2 LLMClient 통합 (mock client)
class _MockClient:
    def __init__(self):
        self.last_ctx = None
    async def generate(self, ctx, *, cache_policy=None):
        self.last_ctx = ctx
        return LLMResponse(
            text='mock-ok',
            static_hash=get_static_hash(ctx),
            cache_hit=False,
            usage=TokenUsage(input_tokens=1, output_tokens=1),
        )
    def count_tokens(self, text):
        return len(text.split())

client = _MockClient()
yesterday = date.today() - timedelta(days=1)
ctx_with_date = RenderContext(last_emit_date=yesterday)
resp = await call_with_attachments(client, ctx_with_date, user_input='안녕')

print(f'mock generate response: {resp.text}')
print(f'전달된 ctx.messages 개수: {len(client.last_ctx.messages)} (1 user + N attachments)')
for i, m in enumerate(client.last_ctx.messages):
    block_summary = [b.type for b in m.content]
    print(f'  [{i}] {m.role} {block_summary}')

## 실습 (4개)

직접 코드 작성해서 동작 확인.

### 실습 1 — 도메인 어태치먼트 만들기

`ctx.tool_pool` 의 도구 수가 5개 이상일 때만 "많은 도구가 활성화되어 있습니다 ..." 메시지 발화하는 어태치먼트를 만들고 register, 호출로 검증.

### 실습 2 — 도구 풀 게이트로 disable

실습 1 의 어태치먼트를 "DebugMode" 도구가 풀에 있으면 자동 비활성하도록 게이트 등록. 두 가지 케이스 (도구 있을 때 / 없을 때) 호출 결과 비교.

### 실습 3 — `count_turns_since` 활용

`count_turns_since` 헬퍼 사용해서 "마지막 user 메시지 이후 assistant turn 5번 지나면 reminder 발화" 어태치먼트 작성. RenderContext 슬롯에 `messages` 다양한 길이로 박아 검증.

### 실습 4 — `smoosh` 활용한 in-loop reminder

ReAct 루프 가정 — 도구 결과 메시지 (ToolResultBlock) 가 마지막에 있는 messages 튜플을 만들고, `smoosh_into_last_tool_result` 로 reminder 합성. 결과의 마지막 메시지 content 가 두 블록 (tool_result + TextBlock) 인지 확인. tool_use_id 보존도 같이.

In [ ]:
# 실습 1 작성 영역
# class _LotsOfTools:
#     ...


In [ ]:
# 실습 2 작성 영역
# register_tool_pool_gate('lots_of_tools', lambda tools: ...)


In [ ]:
# 실습 3 작성 영역
# class _IdleAttachment:
#     async def build(self, ctx):
#         turns = count_turns_since(ctx.messages, lambda m: m.role == 'user')
#         ...


In [ ]:
# 실습 4 작성 영역
# msgs = (
#     Message(role='user', content=(ToolResultBlock(tool_use_id='toolu_42', content='...'),)),
# )
# smooshed = smoosh_into_last_tool_result(msgs, '<system-reminder>...</system-reminder>')
# ...
